# Colab starter — Segmentación de heridas en ratas

Correr las celdas en orden. Este notebook clona el repo de GitHub, monta Google Drive (donde están `Images/` y `Masks/`), instala dependencias, y deja todo listo para llamar a los scripts de `src/`.

**Antes de correr:** en Colab, andá a `Entorno de ejecución -> Cambiar tipo de entorno de ejecución -> GPU` para tener GPU disponible.

In [ ]:
# 1) Clonar el repo (reemplazar por tu URL real una vez que lo subas a GitHub)
REPO_URL = "https://github.com/camichahwan/wound_segmentation.git"
!git clone $REPO_URL
%cd wound_segmentation

In [ ]:
# 2) Instalar dependencias
!pip install -q -r requirements.txt

In [ ]:
# 3) Montar Google Drive (te va a pedir autorización la primera vez)
import os, sys

REPO_DIR = "/content/wound_segmentation"
os.chdir(REPO_DIR)  # por si esta celda se corre sola, sin haber corrido la 1 justo antes
sys.path.insert(0, os.path.join(REPO_DIR, "src"))

from config import mount_drive_if_colab, get_data_root, get_images_dir, get_masks_dir

mount_drive_if_colab()
print("Data root:", get_data_root())
print("Images:", get_images_dir())
print("Masks:", get_masks_dir())

In [ ]:
# 4) Verificar que el emparejamiento imagen-máscara funciona antes de entrenar nada
from dataset import list_paired_and_unpaired, train_val_test_split

paired, unpaired = list_paired_and_unpaired()
print(f"Imágenes con máscara: {len(paired)}")
print(f"Imágenes sin máscara todavía: {len(unpaired)}")

train_samples, val_samples, test_samples = train_val_test_split(paired)
print(f"Train: {len(train_samples)} | Val: {len(val_samples)} | Test: {len(test_samples)}")

In [ ]:
# 5) Entrenar la U-Net preentrenada (rápido, buen primer experimento)
!cd src && python train.py --model pretrained --encoder resnet34 --epochs 40 --batch_size 8 --image_size 512

In [ ]:
# 6) Entrenar la U-Net desde cero (más lenta en converger, dejarla correr más épocas)
!cd src && python train.py --model scratch --epochs 80 --batch_size 8 --image_size 512

In [ ]:
# 7) Comparar ambos modelos en el set de test
!cd src && python evaluate.py --checkpoints ../outputs/checkpoints/unet_pretrained.pt ../outputs/checkpoints/unet_scratch.pt